In [11]:
import pylbsr.notebooks
import pylbsr.misc

import torch
import yaml
from dotmap import DotMap
from pathlib import Path
from parnet_additional_utils import GzListDataset

/home/lhofer/pixi-envs/parnet--unified-7433550039317049837/envs/parnet-dev-cu12/lib/python3.10/site-packages/gin/config.py:615: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  decorated_class = decorating_meta(cls.__name__, (cls,), overrides)
Seed set to 42


In [13]:
_notebook_name = "01_explore_globalclip_data.ipynb"
_notebook_path = f"notebooks/globalclip-head/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")

[09:08:04] INFO - Project directory: /mnt/storage1/workspace/lhofer/parnet--globalclip-head


⏱ 0.01 s (00:00:00)


In [14]:
_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.yaml").read_text())

FILEPATHS = DotMap()
FILEPATHS["data"] = Path(_fp_cfg["global_clip"]["data_lysate_noNHS"])
FILEPATHS["metadata"] = Path(_fp_cfg["global_clip"]["metadata_lysate_noNHS"])
logger.info(f"Data: {FILEPATHS.data}")

[09:08:42] INFO - Data: /mnt/storage1/ml4rg26-deconvgclip/provided_data/600nt_globalCLIP_synchronized_datasets/globalclip_lysate_noNHS_600bp_signalfiltered.pt


⏱ 0.01 s (00:00:00)


In [17]:
# ── Raw tensor structure ───────────────────────────────────────────────────────
data = torch.load(FILEPATHS.data, mmap=True)

def _summarize(v, indent=0):
    pad = "  " * indent
    if isinstance(v, dict):
        for k, val in v.items():
            print(f"{pad}{repr(k)}:")
            _summarize(val, indent + 1)
    elif isinstance(v, list):
        print(f"{pad}list  len={len(v)}")
        if len(v) > 0:
            print(f"{pad}  [0]:")
            _summarize(v[0], indent + 2)
    elif hasattr(v, "shape"):
        print(f"{pad}{type(v).__name__}  shape={tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"{pad}{type(v).__name__}  {str(v)[:120]}")

_summarize(data)

'valid':
  list  len=7361
    [0]:
      'meta':
        'name':
          str  chr2:8729695-8730295:-
        'pad_side':
          int  -1
      'inputs':
        'sequence':
          str  ACCCTGCTCTTAGGGGCTCACCTAGGTGAGTGCACAGCCTGTGACGCTACAGGGAGAGGCTGAGTAAACCGAGATCCAGCGTTCTGTATGGCAGGGGTATTGCTTATCACAGAGGTTCTG
      'outputs':
        'globalCLIP':
          'indices':
            Tensor  shape=(2, 34)  dtype=torch.int64
          'values':
            Tensor  shape=(34,)  dtype=torch.float32
          'size':
            Size  torch.Size([1, 600])
        'control':
          'indices':
            Tensor  shape=(2, 24)  dtype=torch.int64
          'values':
            Tensor  shape=(24,)  dtype=torch.float32
          'size':
            Size  torch.Size([1, 600])
'train':
  list  len=39052
    [0]:
      'meta':
        'name':
          str  chr4:3512611-3513211:-
        'pad_side':
          int  -1
      'inputs':
        'sequence':
          str  TACTGAGCTGCTTTTGTGATTTGAGCAT

In [19]:
# ── Load via GzListDataset and inspect a sample ───────────────────────────────
train_ds = GzListDataset(
    FILEPATHS.data, split="train",
    length=600, total_key="globalCLIP", control_key="control",
    shuffle=True, mmap=True,
)
val_ds = GzListDataset(
    FILEPATHS.data, split="valid",
    length=600, total_key="globalCLIP", control_key="control",
    shuffle=False, mmap=True,
)
logger.info(f"Train size: {len(train_ds)}  |  Val size: {len(val_ds)}")

sample = train_ds[0]
print("\n── sample keys ──")
print("inputs :", list(sample["inputs"].keys()))
print("outputs:", list(sample["outputs"].keys()))
print()
print("── sample shapes ──")
for k, v in sample["inputs"].items():
    print(f"  inputs[{repr(k)}]  {tuple(v.shape)}  {v.dtype}")
for k, v in sample["outputs"].items():
    print(f"  outputs[{repr(k)}]  {tuple(v.shape)}  {v.dtype}")

Loading (mmap) globalclip_lysate_noNHS_600bp_signalfiltered.pt split='train'... loaded 39052 samples.
Loading (mmap) globalclip_lysate_noNHS_600bp_signalfiltered.pt split='valid'... 

[09:22:32] INFO - Train size: 39052  |  Val size: 7361


loaded 7361 samples.

── sample keys ──
inputs : ['sequence']
outputs: ['total', 'control']

── sample shapes ──
  inputs['sequence']  (4, 600)  torch.float32
  outputs['total']  (1, 600)  torch.int64
  outputs['control']  (1, 600)  torch.int64
⏱ 21.63 s (00:00:21)
